In [ ]:
from pydicom import dcmread
ds = dcmread("/data/soin/retina/UKbiobank_90947/dataset_670066/test/1000016_21013_0_0.dcm")

In [ ]:
import pandas as pd

In [ ]:
head=pd.read_csv("/data/soin/octgwas/oct_quality_sauna/result/csv/UKBB_final_headers.csv")

In [ ]:
'Plotting functions & data visualization'
import os
import shutil
import matplotlib.pyplot as plt
import numpy as np
import cv2
import imageio
import seaborn as sns

from compute import *

from scipy.optimize import curve_fit


def plot_quality_distribution (df,  output_file, fig_name=None, bins_=150, log=False):
    """Based on the POC score, plot OCTs quality distribution.
    Args:
        df (DataFrame) : contains the scores of each OCTs
        output_file (str) :  name of the directory to save the plot
        fig_name(str) : name of the figure
        bins_ (int) :  number of bins to plot the histogram.

    """
    #Histogram
    
    palette=sns.color_palette("crest",n_colors=150)
    sns_plot=sns.histplot(df, x="score_displacement_weighted", bins = bins_, color=palette[0])
    sns.despine()
    fig=sns_plot.get_figure()

    # Plot config
    plt.title('Distribution of quality score')
    plt.xlabel(f'Quality score with a mean of {np.round(np.mean(df.score_displacement_weighted),2)} ')
    plt.ylabel('OCT Count')
    if(log): plt.yscale('log')
    
    plt.show()
    

    # Saving
    fig_file=output_file + f"/{fig_name}.png"
    fig.savefig(fig_file, transparent=True)



In [ ]:
plot_quality_distribution(head,"/data/soin/octgwas/oct_quality_sauna/result/", "complete_dst_weighted" )

In [ ]:
import warnings
warnings.filterwarnings("ignore", message="Boolean Series key will be reindexed to match DataFrame index")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 
import pandas as pd
import ast
import json
import os
from sklearn.metrics import f1_score, precision_score, cohen_kappa_score
from configparser import ConfigParser
from argparse import ArgumentParser

def divide_peaks(df, list_index) :
    
    #TODO: Check the len of list_index
    small = df[df.Peaks_Heights<list_index[0]]
    medium =df[df.Peaks_Heights<list_index[1]][df.Peaks_Heights>list_index[0]].reset_index(drop=True)
    high_medium = df[df.Peaks_Heights<list_index[2]][df.Peaks_Heights>=list_index[1]].reset_index(drop=True)
    high =df[df.Peaks_Heights>=list_index[2]].reset_index(drop=True)
    return small, medium, high_medium, high

def get_metrics(df, name, col_sauna, type_metric):
    ##TODO incroprate a type_ arg so that can compute different metrics
    if(type_metric=="F1_Scores"):
        return(f1_score(df[name].values, df[col_sauna].values))
    if(type_metric=="Precision"):
        return(precision_score(df[name].values, df[col_sauna].values))
    if(type_metric=="Kappas_Coefficient"):
        return(cohen_kappa_score(df[name].values, df[col_sauna].values))
    
def extract_unique_dict(dict_): 
    unique_dict = {}
    for key, value_list in dict_.items():
        unique_arrays = [list(x) for x in set(tuple(x) for x in value_list)]
        unique_dict[key] = unique_arrays
    return unique_dict

def extract_metrics_df(csv_files, directory_path, modality,list_,name_types,type_metric):
    my_dictionary = {}
    for csv in csv_files: 
        
        all_ = pd.read_csv(f"{directory_path}/{modality}/{csv}", sep= ",")
        name_1=csv.split("_")[2]
        name_2 = csv.split("_")[3]
        small, medium, high_medium, high=divide_peaks(all_, list_)
        
        ##Can change the list of df and remove all df
        for idx_,df in enumerate([small, medium, high_medium, high]):
            val=get_metrics(df, name_1,name_2,type_metric)
            my_dictionary.setdefault(f"{name_1}_{name_2}", []).append([val, name_types[idx_]])
                
    return extract_unique_dict(my_dictionary)

def drop_col_conflict (df):
    col_to_drop=[]
    for col in df.columns:
        number=int(col.split("_")[2])
        if(number%2 !=0): 
            col_to_drop.append(col)
    return col_to_drop

def convert_ditc_to_pd (dict_):
    df=pd.DataFrame(dict_)
    exploded_df=df.apply(pd.Series.explode, axis=1)
    # Rename columns to avoid conflicts
    exploded_df.columns = [f'{col}_{i}' for i, col in enumerate(exploded_df.columns)]
    # Add a type columns (height Peaks info)
    col_to_drop=drop_col_conflict(exploded_df)
    exploded_df["type"] = exploded_df[exploded_df.columns[1]]
    exploded_df.drop(col_to_drop, axis=1, inplace=True)
    return exploded_df

def array_single_array(array):
    return([item for sublist in array for item in sublist])

def df_to_plot(df):
    names=[]
    values=[]
    types=[]

    for col in (df.drop("type", axis=1).columns): 


        names.append([col.split("_")[0]+"_"+col.split("_")[1]]*len(df[col]))
        values.append(df[col].values)
        types.append(df["type"].values)

    names=array_single_array(names)
    values=array_single_array(values)
    types=array_single_array(types)
    return pd.DataFrame(list(zip(names, values, types)),columns=("Annotators", type_metric, "Peaks size"))

def plot_metric_score(df, modality, outptut_dir, list_separate_peaks, type_metric, names_type_peaks,hue_order):
   
    mean =df.groupby("Peaks size").mean()
    std_mean=df.groupby("Peaks size").sem()
    
    
    palette=sns.color_palette("Set2",n_colors=len(np.unique(df.Annotators)))
    #sns.light_palette("seagreen", reverse=True, n_colors=len(np.unique(df.Annotators)))
    sns.set_theme(style="whitegrid", palette=palette)
    g=sns.catplot(data=df, y=type_metric, x="Peaks size",
                  hue="Annotators", kind="bar", palette=palette, order=names_type_peaks, hue_order=hue_order)

    # Access the underlying Matplotlib axes
    ax = g.ax
    
    # Calculate and plot horizontal bars for the mean value of each group
    plt.ylim([0, 1])
    for i, cat in enumerate(names_type_peaks):

            y=mean.loc[cat].values
            y_std =std_mean.loc[cat].values
            x = i*0.25
            ax.axhline(y, xmin=x, xmax=x+0.24,
                       color="black", linestyle="--", linewidth=1)

            ax.axvline(i-0.49,  ymin= y-y_std , ymax=y+y_std,  color="black", linestyle="--", linewidth=1)
            ax.axvline(i+0.49,  ymin= y-y_std , ymax=y+y_std,  color="black", linestyle="--", linewidth=1)


    
    title=f"{type_metric} bewteen SAUNA and Annotators \n With  modality of {modality} \n split : {list_separate_peaks}"
    plt.title(title)
    plt.xticks(rotation=45)
   
    plt.savefig(f"{outptut_dir}/png/{modality}/{title}.png", bbox_inches="tight", transparent=True)
 

In [ ]:
     
def main (directory_path,names_type_peaks,list_separate_peaks, modality, outptut_dir,type_metric,hue_order):
    
    csv_files = [file for file in os.listdir(f"{directory_path}/{modality}/") if file.endswith('.csv')]
    dic=extract_metrics_df(csv_files, directory_path, modality, list_separate_peaks,names_type_peaks, type_metric)

    df=convert_ditc_to_pd(dic)
    final = df_to_plot(df)
    
    plot_metric_score(final, modality, output_dir, list_separate_peaks,type_metric ,names_type_peaks,hue_order)


In [ ]:
directory_path= "/data/soin/octgwas/SAUNA_VS_HUMAN/results/csv/"
names_type_peaks=["small", "medium", "high_medium", "high"]
list_separate_peaks= [5,10,25]
modality=320
output_dir="/data/soin/octgwas/SAUNA_VS_HUMAN/results/"
type_metric="F1_Scores"
hue_order=['Alicia_Flavie',  'Alicia_Laurent',
       'Flavie_Ilenia', 'Flavie_Laurent',
       'Ilenia_Laurent','Alicia_Ilenia','Consensus1_Consensus2' ]

In [ ]:
main(directory_path,names_type_peaks,list_separate_peaks, modality, output_dir,type_metric, hue_order)

In [ ]:
plot_metric_score(df, modality, output_dir, list_separate_peaks,type_metric ,names_type_peaks,hue_order, val, std)

In [ ]:
mean_type=(df.groupby("type", group_keys=True).mean()).mean(axis=1)


In [ ]:
from scipy.stats import sem
grouped_df = df.groupby('type').mean()

# Calculate the standard error of the mean for each group
sem_values = grouped_df.agg(lambda x: sem(x, nan_policy='omit'), axis=1)
sem_values

## Visualisation of the results (all)

In [ ]:
plot_quality_distribution(df_final,output_file='../result', 
                          fig_name="plot_final_OL", bins_=30)

In [ ]:
plot_quality_distribution(low_df,output_file='../result',
                          fig_name="plot_final_annot", bins_=30, log=True)


In [ ]:
thr=0.9
high_df = df_final[df_final.score >thr]

In [ ]:
plot_quality_distribution(high_df,output_file='../result', 
                          fig_name="plot_final_annot", bins_=40)

In [ ]:
plot_quality_distribution(high_df,output_file='../result', 
                          fig_name="plot_final_annot", bins_=40, log=True)

In [ ]:
x=len(df_final[df_final.score>0.7])

In [ ]:


print(f"There are {x} OCTs with a score above 0.7 and a total of OCTs of {len(df_final)}")

In [ ]:
ukbb=pd.read_csv("/data/soin/octgwas/oct_quality_sauna/result/csv/UKBB_headers_median.csv")

In [ ]:
plot_quality_distribution(ukbb,output_file='../result', fig_name="UKBB_dist", bins_=50)

In [ ]:
OL=pd.read_csv("/data/soin/octgwas/Results_SAUNA_OL/csv/test_OL_headers.csv")

In [ ]:
plot_quality_distribution(OL,output_file='../result', fig_name="OL_dist", bins_=30)

## Comparison Mean and Median for Sauna Score 

In [ ]:
def print_stat(df_final):
    stat_07=len(df_final[df_final["score"]>0.7])
    print(f"There are {stat_07} OCTs with a score above 0.7 and a total of OCTs of {len(df_final)}")

In [ ]:
score_median= pd.read_pickle("../pkl/med_all.pkl")

In [ ]:
plot_quality_distribution(score_median, output_file='../result', fig_name="median", bins_=10)

In [ ]:
print_stat(score_median)

In [ ]:
score_mean= pd.read_pickle("../pkl/mean_all.pkl")
plot_quality_distribution(score_mean, output_file='../result', fig_name="mean", bins_=10)

In [ ]:
print_stat(score_mean)

In [ ]:
sns.scatterplot(x=score_mean.score, y=score_median.score, color="lightblue")

plt.title('Mean vs Median quality Score')

plt.xlabel('Mean score')

plt.ylabel('Median Score')


In [ ]:
sub_df_med_2 = score_median[(score_median.score>0.2)& (score_median.score<0.8)]
sub_df_med_2.set_index("oct_id", inplace = True)
score_mean.set_index("oct_id", inplace = True)
sub_df_mean_2 = score_mean.loc[sub_df_med_2.index]

In [ ]:
sns.scatterplot(x=sub_df_mean_2.score, y=sub_df_med_2.score, color="lightblue")

plt.title('Mean vs Median quality Score using intermediate Med val')

plt.xlabel('Mean score')

plt.ylabel('Median Score')

In [ ]:
data_2 =pd.DataFrame(pd.concat([sub_df_mean_2.score,sub_df_med_2.score ], axis=0))

n = len(sub_df_med_2.score)
type_1_values = ["Mean"] * n
type_2_values = ["Median"] * n

type_ = type_1_values + type_2_values

list1 =np.arange(len(sub_df_mean_2.score)).tolist()
list2 = np.arange(len(sub_df_mean_2.score)).tolist()

scans_ = list1 + list2  # Concatenate the two lists

data_2["scans"]=scans_
data_2["type"]=type_

In [ ]:
sns.scatterplot(data_2, x="score", y="scans", hue="type", palette="mako")

plt.title('Mean vs Median quality Score using intermediate Med val')

In [ ]:
sub_df_mean =score_mean[(score_mean.score>0.2)& (score_mean.score<0.8)]

In [ ]:
sub_df_mean.set_index("oct_id", inplace = True)

In [ ]:
score_median.set_index("oct_id", inplace = True)

In [ ]:
sub_df_med = score_median.loc[sub_df_mean.index]

In [ ]:
sns.scatterplot(x=sub_df_mean.score, y=sub_df_med.score, color="lightblue")

plt.title('Mean vs Median quality Score')

plt.xlabel('Mean score')

plt.ylabel('Median Score')

In [ ]:
data =pd.DataFrame(pd.concat([sub_df_mean.score,sub_df_med.score ], axis=0))

n = len(sub_df_med.score)
type_1_values = ["Mean"] * n
type_2_values = ["Median"] * n

type_ = type_1_values + type_2_values

list1 =np.arange(len(sub_df_mean.score)).tolist()
list2 = np.arange(len(sub_df_mean.score)).tolist()

scans_ = list1 + list2  # Concatenate the two lists

data["scans"]=scans_
data["type"]=type_

In [ ]:
sns.scatterplot(data, x="score", y="scans", hue="type", palette="mako")